# 04. Advanced Visualizations - Departmental Benchmarks & Wait Times
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

Hospital dashboards must guide executives to quick clinical conclusions. In this notebook, we build production-grade, publication-quality visualizations to analyze departmental volumes, ER wait times, and clinical procedual costs.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "cleaned"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(exist_ok=True)

admissions = pd.read_csv(DATA_DIR / "admissions.csv")
billing = pd.read_csv(DATA_DIR / "billing.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")


## 1. Departmental Revenue Breakdown
Let's visualize total billings (charges) versus direct hospital costs across our medical service lines to highlight high-margin vs high-loss departments.


In [ ]:
dept_billing = pd.merge(billing, departments, on='department_id')
revenue_costs = dept_billing.groupby('department').agg(
    total_revenue=('charge_amount', 'sum'),
    total_cost=('cost_amount', 'sum')
).reset_index()

# Plot side-by-side bar chart
x = np.arange(len(revenue_costs['department']))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
rects1 = ax.bar(x - width/2, revenue_costs['total_revenue'] / 1e6, width, label='Charges (Revenue)', color='#2e8b57')
rects2 = ax.bar(x + width/2, revenue_costs['total_cost'] / 1e6, width, label='Hospital Cost', color='#cd5c5c')

ax.set_ylabel('Amount ($ Millions)', fontweight='bold')
ax.set_xlabel('Clinical Department', fontweight='bold', labelpad=10)
ax.set_title('Departmental Financial Performance: Revenue vs Direct Costs', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(revenue_costs['department'], rotation=15)
ax.legend()
plt.tight_layout()
plt.savefig(VISUALS_DIR / "departmental_revenue_vs_cost.png", dpi=150)
plt.show()


## 2. Wait Times by Admission Severity & Type
Let's check how long patients wait in the Emergency Room, segmented by emergency status and disease severity score.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=admissions, x='severity_score', y='wait_minutes', hue='admission_type', palette='Set2')
plt.title("ER Wait Times (Minutes) by Patient Clinical Severity Score", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Patient Severity Score", fontweight='bold', labelpad=10)
plt.ylabel("Wait Time (Minutes)", fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS_DIR / "er_wait_times_by_severity.png", dpi=150)
plt.show()


## 3. Patient Satisfaction by Hospital Department
Let's visualize patient satisfaction surveys across departments to identify potential service gaps.


In [ ]:
satisfaction_surveys = pd.read_csv(DATA_DIR / "satisfaction_surveys.csv")
sat_dept = pd.merge(satisfaction_surveys, departments, on='department_id')

plt.figure(figsize=(10, 5))
sns.violinplot(data=sat_dept, x='department', y='satisfaction_score', palette='Pastel1', hue='department', legend=False)
plt.title("Patient Satisfaction Score Distribution by Medical Department", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Clinical Department", fontweight='bold', labelpad=10)
plt.ylabel("Satisfaction Score (1-10)", fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS_DIR / "satisfaction_score_violin.png", dpi=150)
plt.show()
